In [ ]:
from torch.utils.data import DataLoader
from typing import Iterable
from tqdm import tqdm

import numpy as np
import pandas as pd
import spacy
import torch
import xgboost

import dianna
from dianna import visualization
from dianna.utils.downloader import download
from dianna.utils.tokenizers import SpacyTokenizer

In [ ]:
class_names = ['constitutive', 'regulatory']
model_path = download('inlegal_bert_xgboost_classifier.json', 'model')

In [ ]:
constitutive_statement_0 = "The purchase, import or transport from Syria of crude oil and petroleum products shall be prohibited."
constitutive_statement_1 = "This Decision shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union."
regulatory_statement_0 = "Where observations are submitted, or where substantial new evidence is presented, the Council shall review its decision and inform the person or entity concerned accordingly."
regulatory_statement_1 = "The relevant Member State shall inform the other Member States of any authorisation granted under this Article."
regulatory_statement_2 = "Member States shall cooperate, in accordance with their national legislation, with inspections and disposals undertaken pursuant to paragraphs 1 and 2."

In [ ]:
from transformers import AutoTokenizer, AutoModel
def create_features(texts: list[str], model_tag="law-ai/InLegalBERT") -> torch.Tensor:
    """Create features for a list of texts."""
    max_length = 512
    tokenizer = AutoTokenizer.from_pretrained(model_tag)
    model = AutoModel.from_pretrained(model_tag)

    def process_batch(batch: Iterable[str]):
        cropped_texts = [text[:max_length] for text in batch]
        encoded_inputs = tokenizer(cropped_texts, padding='longest', truncation=True, max_length=max_length,
                                   return_tensors="pt")
        with torch.no_grad():
            outputs = model(**encoded_inputs)
        last_hidden_states = outputs.last_hidden_state
        sentence_features = last_hidden_states.mean(dim=1)
        return sentence_features

    dataloader = DataLoader(texts, batch_size=1)  # batch size of 1 was quickest for my development machine
    features = [process_batch(batch) for batch in tqdm(dataloader, desc=f'Creating features')]
    return np.array(torch.cat(features, dim=0))

In [ ]:
models={}
def classify_texts(texts: list[str], model_path, return_proba: bool = False):
    features = create_features(texts)
    if model_path not in models:
        print(f'Loading model from {model_path}.')
        model = xgboost.XGBClassifier()
        model.load_model(model_path)
        models[model_path] = model

    model = models[model_path]
    if return_proba:
        return model.predict_proba(features)
    return model.predict(features)

In [ ]:
# ensure the tokenizer for english is available
spacy.cli.download('en_core_web_sm')

In [ ]:
class StatementClassifier:
    def __init__(self):
        self.tokenizer = SpacyTokenizer(name='en_core_web_sm')

    def __call__(self, sentences):
        # ensure the input has a batch axis
        if isinstance(sentences, str):
            sentences = [sentences]

        probs = classify_texts(sentences, model_path, return_proba=True)

        return np.transpose([(probs[:, 0]), (1 - probs[:, 0])])

model_runner = StatementClassifier()

## Test the model

The cell below is the one that timed out in CI (job 75236702373).
It has been split one line per cell so we can identify which line hangs.

In [ ]:
statements = [constitutive_statement_0, constitutive_statement_1, regulatory_statement_0, regulatory_statement_1,
               regulatory_statement_2]

In [ ]:
actual_classes = [class_names[c] for c in [0,0,1,1,1]]

In [ ]:
model_outputs = model_runner(statements)

In [ ]:
predictioned_classes = [class_names[m] for m in np.argmax(model_outputs, axis=1)]

In [ ]:
pd.DataFrame({'statement': statements, 'prediction': predictioned_classes, 'actual': actual_classes})